# Telecom Customer Churn Analysis & EDA
### Comprehensive Exploratory Data Analysis & Statistical Testing
**Tech Stack:** Python | Pandas | NumPy | Matplotlib | Seaborn | SciPy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Visual configuration
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 5)
%matplotlib inline

## 1. Load Raw Dataset & Initial Inspection

In [ ]:
raw_df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'Dataset Dimensions: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns')
raw_df.head(5)

## 2. Data Cleaning & Type Conversion
- Convert `TotalCharges` from object to float (handling blank whitespace characters).
- Impute zero-tenure customer charges.
- Check and drop duplicates.
- Create analytical helper columns (`Tenure_Group`, `Churn_Numeric`).

In [ ]:
df = raw_df.copy()
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce').fillna(0.0)
df['SeniorCitizen_Label'] = df['SeniorCitizen'].map({1: 'Yes', 0: 'No'})
df['Churn_Numeric'] = df['Churn'].map({'Yes': 1, 'No': 0})

def get_tenure_group(t):
    if t <= 12: return '0-12 Months'
    elif t <= 24: return '13-24 Months'
    elif t <= 48: return '25-48 Months'
    elif t <= 60: return '49-60 Months'
    else: return '60+ Months'

df['Tenure_Group'] = df['tenure'].apply(get_tenure_group)
df.to_csv('../data/processed/cleaned_churn.csv', index=False)
print('Cleaned dataset saved to data/processed/cleaned_churn.csv')
df.info()

## 3. Exploratory Data Analysis (EDA)
### 3.1 Overall Churn Rate

In [ ]:
churn_counts = df['Churn'].value_counts()
plt.figure(figsize=(6, 4))
sns.barplot(x=churn_counts.index, y=churn_counts.values, palette=['#2b5c8f', '#d95f02'])
plt.title('Overall Customer Churn Distribution', fontsize=12, fontweight='bold')
plt.ylabel('Count')
for i, v in enumerate(churn_counts.values):
    plt.text(i, v/2, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', color='white', fontweight='bold')
plt.show()

### 3.2 Churn by Contract Type & Payment Method

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=df, x='Contract', y='Churn_Numeric', ax=axes[0], ci=None, palette='Blues_r')
axes[0].set_title('Churn Rate by Contract Type', fontweight='bold')
axes[0].set_ylabel('Churn Rate')

sns.barplot(data=df, x='PaymentMethod', y='Churn_Numeric', ax=axes[1], ci=None, palette='Oranges_r')
axes[1].set_title('Churn Rate by Payment Method', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)
axes[1].set_ylabel('Churn Rate')
plt.tight_layout()
plt.show()

## 4. Statistical Hypothesis Testing
### 4.1 Chi-Square Test: Contract Type vs Churn

In [ ]:
chi2, p_val, dof, _ = stats.chi2_contingency(pd.crosstab(df['Contract'], df['Churn']))
print(f'Chi2: {chi2:.4f}, p-value: {p_val:.4e} -> {"Significant" if p_val < 0.05 else "Not Significant"}')

### 4.2 Welch's Two-Sample t-Test: Monthly Charges (Churned vs Retained)

In [ ]:
t_stat, p_val = stats.ttest_ind(
    df[df['Churn']=='Yes']['MonthlyCharges'],
    df[df['Churn']=='No']['MonthlyCharges'],
    equal_var=False
)
print(f't-statistic: {t_stat:.4f}, p-value: {p_val:.4e}')